# LLM Answer Evaluator

Ask a technical question, get an explanation from two models, and see which answer to trust:

- **A cloud model** (OpenAI), streamed in as it is written
- **A local model** running on this computer with Ollama: free, private, works offline

You can choose the explanation **level** (beginner, intermediate, expert), and at the end
the cloud model compares the two answers, so you can see what the local model got right or missed.

In [1]:
# Imports

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Settings

MODEL_CLOUD = "gpt-6-luna"
MODEL_LOCAL = "gemma3:270m"   # better answers: run `ollama pull llama3.2` and change this to "llama3.2"
LEVEL = "beginner"            # beginner, intermediate or expert

In [3]:
# Connect to both models
# Ollama speaks the same API as OpenAI, so one client library works for both:
# only the address (base_url) changes.

load_dotenv(override=True)

cloud = OpenAI()
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # Ollama ignores the key, but the library needs one

In [4]:
# The tutor's instructions

LEVEL_GUIDE = {
    "beginner": "The learner is new to programming. Avoid jargon, or explain it the first time you use it. Use a simple everyday analogy.",
    "intermediate": "The learner knows the basics of the language. Focus on how and why it works, not on basic syntax.",
    "expert": "The learner is experienced. Be concise and precise; mention performance, edge cases and alternatives.",
}

def system_prompt_for(level):
    return f"""
You are a patient, precise technical tutor.
{LEVEL_GUIDE[level]}

Answer in markdown with these sections:
1. **In one sentence** - the short answer
2. **Step by step** - break the code or concept into its parts and explain each one
3. **Example** - a small runnable example with its output
4. **Watch out** - common mistakes or surprises

If the question contains code, explain that exact code, not a general topic.
If you are not sure about something, say so instead of guessing.
"""

In [5]:
# Ask a model and show the answer as it streams in

def ask(client, model, question, level=LEVEL):
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt_for(level)},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    answer = ""
    handle = display(Markdown(f"*{model} is thinking...*"), display_id=True)
    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        update_display(Markdown(answer), display_id=handle.display_id)
    return answer

## Your question

Type over the text below to ask something new, then run the cells underneath.

In [6]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

## Answer from the cloud model

In [7]:
cloud_answer = ask(cloud, MODEL_CLOUD, question)

## In one sentence

This code yields each **distinct, truthy author** found in `books`, using a set to remove duplicates.

## Step by step

```python
yield from {book.get("author") for book in books if book.get("author")}
```

- `for book in books` visits each item in `books`.
- `book.get("author")` looks up the `"author"` value. If the key is missing, `.get()` returns `None` rather than raising an error.
- `if book.get("author")` keeps only truthy values, such as non-empty strings. It skips values like `None` and `""`.
- `{...}` makes a **set**: like a collection that keeps only one copy of each value. So duplicate authors are removed.
- `yield from` yields each value from that set one at a time. It can only be used inside a generator function (a function that uses `yield`).

Because sets do not promise a particular order, the authors may be yielded in any order.

## Example

```python
def authors_from(books):
    yield from {book.get("author") for book in books if book.get("author")}


books = [
    {"title": "Book A", "author": "Sam"},
    {"title": "Book B", "author": "Lee"},
    {"title": "Book C", "author": "Sam"},
    {"title": "Book D", "author": ""},
    {"title": "Book E"},
]

print(sorted(authors_from(books)))
```

Output:

```text
['Lee', 'Sam']
```

`"Sam"` appears only once, and the empty or missing authors are skipped. `sorted()` is used here to make the example’s output order predictable.

## Watch out

- A set removes duplicates **and does not preserve a reliable order**.
- The values must be usable in a set. Strings are fine, but an unhashable value such as a list used as an author would cause an error.
- The condition skips all falsy author values, not just missing ones—for example, it also skips `""`.

## Answer from the local model

Needs Ollama running (`ollama serve`, or the Ollama app). Small local models are fast and free,
but their answers can be less accurate; that is what the comparison below is for.

In [8]:
local_answer = ask(local, MODEL_LOCAL, question)

Okay, I understand. I will provide a code solution and explain the reasoning behind the answer, avoiding jargon or complex sentence structures as much as possible. Let's begin with the code snippet.


## Compare the two answers

The cloud model reviews both answers: what each got right, any mistakes, and which one to trust.

In [9]:
compare_system_prompt = """
You are reviewing two explanations of the same technical question, written for a learner.
Respond in markdown, briefly:
- **Verdict**: which explanation is more correct and more useful, in one sentence
- **Mistakes**: any factual errors in either answer (quote them), or "None found"
- **Missing**: anything important that both answers left out
Be fair and specific. Do not rewrite the answers.
"""

def compare(question, cloud_answer, local_answer):
    prompt = (
        f"Question:\n{question}\n\n"
        f"Answer A ({MODEL_CLOUD}):\n{cloud_answer}\n\n"
        f"Answer B ({MODEL_LOCAL}):\n{local_answer}"
    )
    response = cloud.chat.completions.create(
        model=MODEL_CLOUD,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": compare_system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    display(Markdown(response.choices[0].message.content))

compare(question, cloud_answer, local_answer)

- **Verdict**: Answer A is much more correct and useful: it explains the filtering, duplicate removal, and yielding behavior; Answer B does not answer the question.
- **Mistakes**: Answer A says sets “do not promise a particular order,” which is broadly right, though more precisely set iteration order is not guaranteed by the language. No other factual errors found. Answer B provides no explanation or solution.
- **Missing**: Neither explicitly notes that the set comprehension is evaluated before `yield from` begins, so the authors are collected and deduplicated eagerly rather than yielded as each book is processed.